# Projet BGES d’une Organisation

In [8]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

In [9]:
pip install pyspark


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Users/alixaubert/Documents/NF26/Exercices_TD/.venv-global/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import numpy as np
from datetime import datetime, date, timedelta

import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import Row
from numpy._core.multiarray import empty_like
from sklearn.linear_model import LinearRegression

In [11]:
spark = SparkSession.builder.getOrCreate()

In [12]:
#import zipfile

#ef unzip_data(filename):
    #Unzips filename into the current working directory.
    #Args:
    #filename (str): a filepath to a target zip folder to be unzipped.
    #zip_ref = zipfile.ZipFile(filename, "r")
    #zip_ref.extractall()
    #zip_ref.close()

#unzip_data("BDD_BGES.zip")

In [13]:
ps.set_option('compute.fail_on_ansi_mode', False)
from pyspark.sql.functions import *

In [14]:
BASE_DIR = "BDD_BGES"
SITES = ["LONDON", "BERLIN", "NEWYORK", "LOSANGELES","PARIS", "SHANGHAI"]

Traduction des données / Normalisation de langue

In [15]:
JOB_TRANSLATIONS = {
    'Ingénieur Data': 'Data Engineer', 'Data Engineer': 'Data Engineer',
    'Ingénieur Informaticien': 'IT Engineer', 'IT Engineer': 'IT Engineer',
    'Cadre': 'Manager', 'Manager': 'Manager',
    'Economiste': 'Economist', 'Economist': 'Economist',
    'DRH': 'HR Director', 'HR Director': 'HR Director'
}

MISSION_TRANSLATIONS = {
    'Conférence': 'Conference', 'Conference': 'Conference',
    'Réunion': 'Meeting', 'Meeting': 'Meeting',
    'Rencontre entreprises': 'Business Meeting', 'Business Meeting': 'Business Meeting',
    'Formation': 'Training', 'Training': 'Training',
    'Développement': 'Development', 'Development': 'Development'
}

#Librairie de trad python?
# !!!! Les espaces dans les noms de fonctions
# FR/ Al à trad
def normalize_data(df, col_name, translation_dict):
    """Normalise les langues des secteurs d'activité et des missions."""
    if col_name in df.columns:
        df[col_name] = df[col_name].map(translation_dict).fillna(df[col_name])
    return df

Fonctions pour les missing values

In [16]:
def handle_missing_values(df, strategy="mean", target_col=None, feature_cols=None):
    """Complète les infos manquantes par moyenne ou régression linéaire"""
    if df.empty:
        return df

    if strategy == "mean" and target_col:
        df[target_col] = df[target_col].fillna(df[target_col].mean())

    elif strategy == "regression" and target_col and feature_cols:
        # Isolation des données incomplètes
        train_data = df.dropna(subset=feature_cols + [target_col])
        missing_data = df[df[target_col].isnull() & df[feature_cols].notnull().all(axis=1)]

        if not missing_data.empty and not train_data.empty:
            X_train = train_data[feature_cols]
            y_train = train_data[target_col]
            X_missing = missing_data[feature_cols]

            model = LinearRegression()
            model.fit(X_train, y_train)
            df.loc[missing_data.index, target_col] = model.predict(X_missing)

    return df

def standardize_timezone(df, date_col):
    """Met à jour le fuseau horaire vers UTC."""
    if date_col in df.columns:
        # Convertir en datetime, puis forcer en UTC
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce', utc=True)
    return df

In [17]:
def etl(current_date):

    date_str = current_date.strftime("%Y%m%d")
    print(f"--- Lancement ETL pour le jour : {date_str} ---")

    # Dictionnaires pour stocker les données du transformer
    missions_jour = []
    info_jour = []

    for site in SITES:
        # --- EXTRACTOR
        # Chemins d'accès
        mission_path = os.path.join(BASE_DIR, f"BDD_BGES_{site}", f"BDD_BGES_{site}_MISSION", f"MISSION_{date_str}.txt")
        it_path = os.path.join(BASE_DIR, f"BDD_BGES_{site}", f"BDD_BGES_{site}_INFORMATIQUE", f"MATERIEL_INFORMATIQUE_{date_str}.txt")

        # Extraction Missions
        if os.path.exists(mission_path):
            df_mission = pd.read_csv(mission_path, sep='\t')

            # --- TRANSFORMER
            # Supprimer les doublons et colonnes inutiles
            df_mission = df_mission.drop_duplicates()

            # Normalisation
            df_mission_norm = normalize_data(df_mission, 'TYPE MISSION', MISSION_TRANSLATIONS)
            df_mission_norm = standardize_timezone(df_mission, 'DATE MISSION')
            #Détecter les anomalies ici?
            #Traitement des informations erronées ou manquantes ici

            # Flagging erreurs potentielles ? (ex: transport inconnu)
            #transports_valides = ['Avion', 'Train', 'Taxi', 'Transports en commun']
            #df_mission['Erreur_Transport'] = ~df_mission['TRANSPORT'].isin(transports_valides)

            missions_jour.append(df_mission)

        # Extraction Matériel Informatique
        if os.path.exists(it_path):
            df_it = pd.read_csv(it_path, sep='\t')

            # Imputation des valeurs manquantes
            #A compléter
            if '' in df_it.columns and '' in df_it.columns:
                df_it = handle_missing_values(df_it, strategy="regression", target_col="", feature_cols=[""])

            info_jour.append(df_it)

    # --- LOAD ---
    # Chargement dans la BDD finale, pas encore mis en place

    if missions_jour or info_jour:
        #df_all_missions = pd.concat(missions_jour, ignore_index=True)
        #load_to_snowflake_db(df_all_missions, table="Fait_Mission")
        #print(f"Chargement de {len(df_all_missions)} missions dans le Data Warehouse.")
        pass
    return

In [18]:
def ask_question(num: int):
  #Mettre toutes les mesures à la même metrique
  pass

In [ ]:
def main():
  """Initialise le Data Warehouse et itère sur l'intervalle de temps."""
  print("Projet BGES - Initialisation du Data Warehouse")

  # Chargement du fichier materiel_informatique_impact.csv (Dim_Materiel)
  impact_csv_path = os.path.join(BASE_DIR, "materiel_informatique_impact.csv")
  if os.path.exists(impact_csv_path):
      df_impact = pd.read_csv(impact_csv_path)
  #mettre toutes les mesures à la même metrique pour les calculs

  # Chargement du personnel pour chaque site (Dim_Personnel)
  for site in SITES:
      pers_path = os.path.join(BASE_DIR, f"BDD_BGES_{site}", f"PERSONNEL_{site}.txt")
      if os.path.exists(pers_path):
          df_pers = pd.read_csv(pers_path, sep='\t')
          df_pers = normalize_data(df_pers, 'FONCTION PERSONNEL', JOB_TRANSLATIONS)

  #Création du modèle étoile

  print("Initialisation des tables de dimensions terminée.\n")

  # Définition de l'intervalle de dates
  start_date = datetime(2026, 4, 29)
  end_date = datetime(2026, 11, 14)
  delta = timedelta(days=1)

  current_date = start_date

  # Mode de chargement des nouvelles données dans le modèle étoile ici
  def load_to_snowflake_db(new_missions : str, new_informatique : str) :
    """Chargement des données dans la base de données Snowflake."""
    #TODO

  # Boucle sur les jours
  while current_date <= end_date:
      print(f"\nLancement du processus ETL pour le jour : {current_date.strftime('%Y-%m-%d')}")
      # On appelle l'ETL pour le jour N
      empty_like(current_date)

      # Poser une question?
      rep = str(input("Souhaitez-vous poser une question? Si oui, laquelle? (Sinon, 0)"))
      if rep !=0:
          ask_question(rep)
      print("\nOn passe au jour suivant!")

      current_date += delta

  print("Processus terminé avec succès.")

if __name__ == "__main__":
    main()

Projet BGES - Initialisation du Data Warehouse
Initialisation des tables de dimensions terminée.


Lancement du processus ETL pour le jour : 2026-04-29

On passe au jour suivant!

Lancement du processus ETL pour le jour : 2026-04-30

On passe au jour suivant!

Lancement du processus ETL pour le jour : 2026-05-01

On passe au jour suivant!

Lancement du processus ETL pour le jour : 2026-05-02

On passe au jour suivant!

Lancement du processus ETL pour le jour : 2026-05-03

On passe au jour suivant!

Lancement du processus ETL pour le jour : 2026-05-04

On passe au jour suivant!

Lancement du processus ETL pour le jour : 2026-05-05

On passe au jour suivant!

Lancement du processus ETL pour le jour : 2026-05-06

On passe au jour suivant!

Lancement du processus ETL pour le jour : 2026-05-07

On passe au jour suivant!

Lancement du processus ETL pour le jour : 2026-05-08


In [1]:

# ==============================================================================
# CONFIGURATION (codé en dur)
# ==============================================================================

BASE_DIR = os.path.join(os.path.dirname("BDD_BGES"), "BDD_BGES")

SITES = ["BERLIN", "LONDON", "LOSANGELES", "NEWYORK", "PARIS", "SHANGHAI"]

IMPACT_PATH = os.path.join(BASE_DIR, "materiel_informatique_impact.csv")

# Colonnes à conserver par type de fichier
COLS_PERSONNEL = [
    "ID_PERSONNEL", "NOM_PERSONNEL", "PRENOM_PERSONNEL",
    "NUM_VOIE", "CMPL_VOIE", "CD_POSTAL", "VILLE", "PAYS",
    "FONCTION_PERSONNEL", "TS_CREATION_PERSONNEL", "TS_MAJ_PPERSONNEL"
]

COLS_MATERIEL = [
    "ID_MATERIELINFO", "ID_PERSONNEL", "NOM_PERSONNEL", "PRENOM_PERSONNEL",
    "DATE_ACHAT", "TYPE", "MODELE"
]

COLS_MISSION = [
    "ID_MISSION", "ID_PERSONNEL", "NOM_PERSONNEL", "PRENOM_PERSONNEL",
    "DATE_MISSION", "TYPE_MISSION", "VILLE_DEPART", "PAYS_DEPART",
    "VILLE_DESTINATION", "PAYS_DESTINATION", "TRANSPORT", "ALLER_RETOUR"
]

# ==============================================================================
# SCHÉMA FLOCON — initialisé dans le main, réutilisé dans etl()
# ==============================================================================

schema = {
    # Table de faits centrale
    "ALICIA_KEYS": pd.DataFrame(columns=[
        "ID_PERSONNEL", "ID_MATERIELINFO", "ID_MISSION"
    ]),
    # Dimensions
    "DF_PERSONNEL": pd.DataFrame(columns=COLS_PERSONNEL),
    "DF_MATERIEL":  pd.DataFrame(columns=COLS_MATERIEL + ["IMPACT"]),
    "DF_MISSION":   pd.DataFrame(columns=COLS_MISSION),
}

# ==============================================================================
# HELPERS
# ==============================================================================

def _filter_columns(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Supprime toutes les colonnes qui ne sont pas dans 'cols'."""
    existing = [c for c in cols if c in df.columns]
    missing  = [c for c in cols if c not in df.columns]
    if missing:
        print(f"    [WARN] Colonnes attendues mais absentes : {missing}")
    return df[existing].copy()


def _append_unique(df_existing: pd.DataFrame, df_new: pd.DataFrame, pk: str) -> pd.DataFrame:
    """Ajoute df_new dans df_existing en dédupliquant sur la clé primaire 'pk'."""
    if df_new.empty:
        return df_existing
    combined = pd.concat([df_existing, df_new], ignore_index=True)
    return combined.drop_duplicates(subset=[pk], keep="last")


def normalize_data(df, col, translations):
    """Normalise les valeurs d'une colonne via un dictionnaire de traduction."""
    if col in df.columns:
        df[col] = df[col].map(translations).fillna(df[col])
    return df


def standardize_timezone(df, col):
    """Convertit une colonne de dates en datetime standardisé."""
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
    return df


def handle_missing_values(df, strategy="mean", target_col="", feature_cols=None):
    """Imputation simple des valeurs manquantes (à enrichir selon les besoins)."""
    if strategy == "mean" and target_col in df.columns:
        df[target_col] = df[target_col].fillna(df[target_col].mean())
    return df


# Dictionnaires de traductions (à compléter)
MISSION_TRANSLATIONS = {
    # ex : "business trip": "Voyage d'affaires"
}

# ==============================================================================
# FONCTION ETL
# ==============================================================================

def etl(current_date):
    date_str = current_date.strftime("%Y%m%d")
    print(f"--- Lancement ETL pour le jour : {date_str} ---")

    # Chargement du référentiel IMPACT une seule fois
    impact_ref = pd.read_csv(IMPACT_PATH, sep=";", dtype=str)
    impact_ref.columns = [c.strip().upper() for c in impact_ref.columns]
    impact_ref = impact_ref[["TYPE", "MODELE", "IMPACT"]]
    print(f"[OK] Référentiel IMPACT chargé ({len(impact_ref)} entrées)")


    # Dictionnaires pour stocker les données du transformer
    missions_jour  = []
    info_jour      = []
    personnel_jour = []

    for site in SITES:

        # ── EXTRACTOR ─────────────────────────────────────────────────────────

        mission_path   = os.path.join(BASE_DIR, f"BDD_BGES_{site}", f"BDD_BGES_{site}_MISSION",
                                      f"MISSION_{date_str}.txt")
        it_path        = os.path.join(BASE_DIR, f"BDD_BGES_{site}", f"BDD_BGES_{site}_INFORMATIQUE",
                                      f"MATERIEL_INFORMATIQUE_{date_str}.txt")
        personnel_path = os.path.join(BASE_DIR, f"BDD_BGES_{site}",
                                      f"PERSONNEL_{site}.txt")

        print(f"\n  [SITE : {site}]")

        # ── MISSION ───────────────────────────────────────────────────────────
        if os.path.exists(mission_path):
            df_mission = pd.read_csv(mission_path, sep='\t')

            # --- TRANSFORMER ---
            df_mission = df_mission.drop_duplicates()

            # Suppression des colonnes inutiles
            df_mission = _filter_columns(df_mission, COLS_MISSION)

            # Normalisation
            df_mission = normalize_data(df_mission, 'TYPE_MISSION', MISSION_TRANSLATIONS)
            df_mission = standardize_timezone(df_mission, 'DATE_MISSION')

            # Flagging transports invalides (décommentez pour activer)
            # transports_valides = ['Avion', 'Train', 'Taxi', 'Transports en commun']
            # df_mission['Erreur_Transport'] = ~df_mission['TRANSPORT'].isin(transports_valides)

            missions_jour.append(df_mission)
            print(f"    [OK] MISSION : {len(df_mission)} ligne(s)")
        else:
            print(f"    [SKIP] Pas de fichier MISSION pour ce jour")

        # ── MATÉRIEL INFORMATIQUE ─────────────────────────────────────────────
        if os.path.exists(it_path):
            df_it = pd.read_csv(it_path, sep='\t')

            # --- TRANSFORMER ---
            df_it = df_it.drop_duplicates()

            # Suppression des colonnes inutiles
            df_it = _filter_columns(df_it, COLS_MATERIEL)

            # Jointure avec le référentiel IMPACT (clé : TYPE + MODELE)
            df_it = df_it.merge(impact_ref, on=["TYPE", "MODELE"], how="left")
            nb_sans_impact = df_it["IMPACT"].isna().sum()
            if nb_sans_impact:
                print(f"    [WARN] {nb_sans_impact} ligne(s) sans correspondance IMPACT")

            # Imputation des valeurs manquantes (à compléter)
            #if '' in df_it.columns and '' in df_it.columns:
                #df_it = handle_missing_values(df_it, strategy="regression",
                                              #target_col="", feature_cols=[""])

            info_jour.append(df_it)
            #print(f"    [OK] MATERIEL : {len(df_it)} ligne(s)")
        #else:
           # print(f"    [SKIP] Pas de fichier MATERIEL pour ce jour")

        # ── PERSONNEL ─────────────────────────────────────────────────────────
        if os.path.exists(personnel_path):
            df_pers = pd.read_csv(personnel_path, sep='\t')

            # --- TRANSFORMER ---
            df_pers = df_pers.drop_duplicates()

            # Suppression des colonnes inutiles
            df_pers = _filter_columns(df_pers, COLS_PERSONNEL)

            personnel_jour.append(df_pers)
            print(f"    [OK] PERSONNEL : {len(df_pers)} ligne(s)")
        else:
            print(f"    [SKIP] Pas de fichier PERSONNEL pour ce site")

    # ── LOAD — Chargement dans le schéma flocon ────────────────────────────────

    if missions_jour or info_jour or personnel_jour:

        # Concaténation de tous les sites du jour
        df_all_missions  = pd.concat(missions_jour,  ignore_index=True) if missions_jour  else pd.DataFrame()
        df_all_materiel  = pd.concat(info_jour,      ignore_index=True) if info_jour      else pd.DataFrame()
        df_all_personnel = pd.concat(personnel_jour, ignore_index=True) if personnel_jour else pd.DataFrame()

        if not df_all_missions.empty:
            schema["DF_MISSION"] = _append_unique(schema["DF_MISSION"], df_all_missions, pk="ID_MISSION")
            print(f"\n[LOAD] DF_MISSION    ← {len(df_all_missions)} ligne(s) insérée(s)")

        if not df_all_materiel.empty:
            schema["DF_MATERIEL"] = _append_unique(schema["DF_MATERIEL"], df_all_materiel, pk="ID_MATERIELINFO")
            print(f"[LOAD] DF_MATERIEL   ← {len(df_all_materiel)} ligne(s) insérée(s)")

        if not df_all_personnel.empty:
            schema["DF_PERSONNEL"] = _append_unique(schema["DF_PERSONNEL"], df_all_personnel, pk="ID_PERSONNEL")
            print(f"[LOAD] DF_PERSONNEL  ← {len(df_all_personnel)} ligne(s) insérée(s)")

        # Table de faits ALICIA_KEYS — croisement des clés du jour
        fact_rows = []
        ids_pers = df_all_personnel["ID_PERSONNEL"].unique() if not df_all_personnel.empty else []

        for id_p in ids_pers:
            ids_mat = (
                df_all_materiel.loc[df_all_materiel["ID_PERSONNEL"] == id_p, "ID_MATERIELINFO"]
                .unique().tolist()
            ) if not df_all_materiel.empty else [None]

            ids_mis = (
                df_all_missions.loc[df_all_missions["ID_PERSONNEL"] == id_p, "ID_MISSION"]
                .unique().tolist()
            ) if not df_all_missions.empty else [None]

            for id_m in ids_mat:
                for id_s in ids_mis:
                    fact_rows.append({
                        "ID_PERSONNEL":    id_p,
                        "ID_MATERIELINFO": id_m,
                        "ID_MISSION":      id_s,
                    })

        if fact_rows:
            df_facts = pd.DataFrame(fact_rows)
            schema["ALICIA_KEYS"] = pd.concat(
                [schema["ALICIA_KEYS"], df_facts], ignore_index=True
            ).drop_duplicates(
                subset=["ID_PERSONNEL", "ID_MATERIELINFO", "ID_MISSION"], keep="last"
            )
            print(f"[LOAD] ALICIA_KEYS   ← {len(df_facts)} ligne(s) insérée(s)")

    # Résumé
    print(f"\n{'─'*50}")
    print("Schéma flocon après ETL :")
    for table, df in schema.items():
        print(f"  {table:<20} : {len(df):>6} ligne(s) au total")
    print(f"{'─'*50}\n")

    return


# ==============================================================================
# MAIN
# ==============================================================================

if __name__ == "__main__":

    # Schéma flocon déjà initialisé en haut du fichier (variable globale `schema`)

    # Date de traitement
    current_date = datetime.today()
    # current_date = datetime(2026, 11, 5)  # ← décommenter pour une date fixe

    # Lancement ETL
    etl(current_date)

NameError: name 'os' is not defined

Proposer de repondre aux questions listées avant de passer au jour suivant